# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [1]:
import re
from urllib.parse import urljoin
import bs4
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup


def scrape_books(min_rating=4.0, max_price=20.0):
  """Scrapes books from 'Books to Scrape' matching a minimum rating and maximum price threshold.

  Parameters:
  -----------
  min_rating : float
      The minimum star rating (1.0 to 5.0) to filter books.
  max_price : float
      The maximum price in GBP (£) to filter books.

  Returns:
  --------
  pd.DataFrame
      A DataFrame containing: UPC, Title, Price (£), Rating, Genre, Availability, Description
  """
  base_url = 'http://books.toscrape.com/'
  current_url = 'http://books.toscrape.com/catalogue/page-1.html'

  rating_map = {'One': 1.0, 'Two': 2.0, 'Three': 3.0, 'Four': 4.0, 'Five': 5.0}

  scraped_books = []

  print(
      f'Starting scraping for books with Rating >= {min_rating} and Price <='
      f' £{max_price}...\n'
  )

  while current_url:
    response = requests.get(current_url)
    if response.status_code != 200:
      print(f'Failed to retrieve page: {current_url}')
      break

    soup = BeautifulSoup(response.content, 'html.parser')
    book_cards = soup.find_all('article', class_='product_pod')

    for card in book_cards:
      # 1. Extracting the Rating
      rating_class = card.find('p', class_='star-rating')['class']
      # # The class next to the 'star-rating' contains the text value (e.g., 'Four')
      rating_word = [c for c in rating_class if c != 'star-rating'][0]
      rating = rating_map.get(rating_word, 0.0)

      # 2. Extracting the price
      price_text = card.find('p', class_='price_color').text
      # Removing the '£' symbol and converting to float
      price = float(re.sub(r'[^0-9.]', '', price_text))

      # FILTERING: load only the subpage if it meets with the conditions!
      if rating >= min_rating and price <= max_price:
        # Relative URL to absolute URL
        rel_link = card.find('h3').find('a')['href']
        detail_url = urljoin(current_url, rel_link)

        # Retrieving the detailed page for a book
        detail_res = requests.get(detail_url)
        if detail_res.status_code != 200:
          continue

        detail_soup = BeautifulSoup(detail_res.content, 'html.parser')

        # Title
        title = detail_soup.find('h1').text.strip()

        # Extracting UPC & Availability from the <table> element
        product_info = {}
        table_rows = detail_soup.find_all('tr')
        for row in table_rows:
          header = row.find('th').text.strip()
          value = row.find('td').text.strip()
          product_info[header] = value

        upc = product_info.get('UPC', 'N/A')
        availability = product_info.get('Availability', 'N/A')

        # Extracting Genre from the breadcrumb navigation
        breadcrumb = detail_soup.find('ul', class_='breadcrumb')
        genre = (
            breadcrumb.find_all('li')[2].text.strip() if breadcrumb else 'N/A'
        )

        # Extracting the Description (if it is presented)
        desc_div = detail_soup.find('div', id='product_description')
        if desc_div:
          description = desc_div.find_next_sibling('p').text.strip()
        else:
          description = 'N/A'

        # Save record
        scraped_books.append({
            'UPC': upc,
            'Title': title,
            'Price (£)': price,
            'Rating': rating,
            'Genre': genre,
            'Availability': availability,
            'Description': description,
        })

    # Look for the 'Next' button for the next page
    next_button = soup.find('li', class_='next')
    if next_button:
      next_rel_link = next_button.find('a')['href']
      current_url = urljoin(current_url, next_rel_link)
    else:
      current_url = None  # Nincs több oldal

  # Create DataFrame 
  df = pd.DataFrame(scraped_books)

  # Create an empty DataFrame with the appropriate columns, if the result is empty
  expected_columns = [
      'UPC',
      'Title',
      'Price (£)',
      'Rating',
      'Genre',
      'Availability',
      'Description',
  ]
  if df.empty:
    df = pd.DataFrame(columns=expected_columns)
  else:
    df = df[expected_columns]

  print(f'Scraping complete! Found {len(df)} matching books.')
  return df

In [3]:
# Execution with the specified parameters
df_filtered_books = scrape_books(min_rating=4.0, max_price=20.0)

# Check the result
display(df_filtered_books.head())
print("\nDataFrame információk:")
df_filtered_books.info()

Starting scraping for books with Rating >= 4.0 and Price <= £20.0...

Scraping complete! Found 75 matching books.


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5.0,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5.0,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5.0,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4.0,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4.0,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."



DataFrame információk:
<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UPC           75 non-null     str    
 1   Title         75 non-null     str    
 2   Price (£)     75 non-null     float64
 3   Rating        75 non-null     float64
 4   Genre         75 non-null     str    
 5   Availability  75 non-null     str    
 6   Description   75 non-null     str    
dtypes: float64(2), str(5)
memory usage: 107.4 KB


In [6]:
# Execution with different parameters
df_filtered_books = scrape_books(min_rating=5.0, max_price=60.0)

# Check the result
display(df_filtered_books.head())
print("\nDataFrame információk:")
df_filtered_books.info()

Starting scraping for books with Rating >= 5.0 and Price <= £60.0...

Scraping complete! Found 196 matching books.


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,4165285e1663650f,Sapiens: A Brief History of Humankind,54.23,5.0,History,In stock (20 available),From a renowned historian comes a groundbreaki...
1,ce6396b0f23f6ecc,Set Me Free,17.46,5.0,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
2,3b1c02bac2a429e6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5.0,Sequential Art,In stock (19 available),Scott Pilgrim's life is totally sweet. He's 23...
3,a34ba96d4081e6a4,Rip it Up and Start Again,35.02,5.0,Music,In stock (19 available),"Punk's raw power rejuvenated rock, but by the ..."
4,c2e46a2ee3b4a322,Chase Me (Paris Nights #2),25.27,5.0,Romance,In stock (19 available),"A Michelin two-star chef at twenty-eight, Viol..."



DataFrame információk:
<class 'pandas.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UPC           196 non-null    str    
 1   Title         196 non-null    str    
 2   Price (£)     196 non-null    float64
 3   Rating        196 non-null    float64
 4   Genre         196 non-null    str    
 5   Availability  196 non-null    str    
 6   Description   196 non-null    str    
dtypes: float64(2), str(5)
memory usage: 322.1 KB
